In [ ]:
from hyperopt import hp, tpe, fmin, STATUS_OK, Trials
from sklearn.metrics import roc_auc_score, matthews_corrcoef, make_scorer, plot_roc_curve 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.model_selection import RepeatedStratifiedkFold
from sklearn.calibration import Calibrated ClassifierCV, calibration_curve
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np 11 import math
import sklearn
import xgboost as xgb 14 import lightgbm as lgbm
from sklearn.ensemble import Gradient BoostingClassifier as gbc
from functools import partial 17 import matplotlib.pyplot as plt
from matplotlib.pyplot import ylim 19 import matplotlib.patches as mpatch
from matplotlib.patches import FancyBboxPatch
import seaborn as sns 22 from sklearn.metrics import classification_report, confusion_matrix
import joblib

import warnings
warnings.filterwarnings ("ignore")


class Binary_classif_model_optmizer:
    
    def init (self, models_dic, general_params): 
        self.models_dic = models_dic
        self.general params = general params        
        self.iteracao = 0
        self.best_loss = 1
        self.best_iter = 0
        
    def reinit counters (self):
        self.iteracao = 0
        self.best_loss = 1 
        self.best_iter = 0
        
    #IMPORTANDO DADOS E DEFININDO CONJUNTOS DE TREINO, VALIDACAO E TESTE
    def initialize_data(self, gen_params, path):
        df_leads_ger = pd.read_csv(path, delimiter=',')
        y = df_leads_ger['DS_STATUS']
        X = df leads_ger[['CANAIS_COUNT', 'CLIENTE_DISPOSITIVO_ACESSO_MODELO',\
                          'CLIENTE_EMAIL_CONTAINS_NOME',\
                          'CLIENTE_EMAIL_CONTAINS UPPERCASE',\ 
                          'CLIENTE_EMAIL_DOMINIO_PULBICO',\
                          'LEADS_COUNT',\
                          'LIMITE_VCL',\
                          'LIMITE_VCL_ITCV_MONT_CV',\
                          'LIMITE_VCL_REDE_MONT_CV',\
                          'QUALIFICADORA_PERIODO_COMPRA',\
                          'ULTIMO_LEAD_CLIENTE_SEGMENTO',\
                          'ULTIMO_LEAD_CLIENT_PREANALYSIS_CORRENTISTA',\ 
                          'ULTIMO_LEAD_ENQUADRADO_PA_ITCV',\
                          'ULTIMO_LEAD_ENQUADRADO_PA_REDE',\
                          'ULTIMO_LEAD_ENQUADRADO_PA_REDE MONTANTE',\
                          'ULTIMO_LEAD_PORCENTAGEM_ENTRADA',\
                          'ULTIMO_LEAD_VALOR_PARCELA',\ 
                          'ULTIMO_LEAD_VALOR_VEICULO',\
                          'ULTIMO_LEAD_VEICULO_ANO',\
                          'ULTIMO_LEAD_VEICULO_ZERO_KM',\ 
                          'GH_AVG_PROSPEC',\
                          'SUBPOP_AVG',\
                          'GH_C2BA',\
                          'GH_BVS', \
                          'DS_SEGMENTO',\
                          'CLIENTE_RENDA_QTD_SALARIO MINIMO', \
                          'PRE_ANALISIS_GH',\
                          'PRE_ANALISIS_GP',\
                          'PRE_ANALISIS_MAX_PARCELA_VALOR',\
                          'PRE_ANALISIS_PRE_ANALISADO',\
                          'DAYS_prospec',\
                          'HAS_EXTERNAL_PARAMS',\
                          'ENVIO_CRM_TRIGGER',\
                          'ENVIO CRM TRIGGER_EMAIL_ABANDONO',\
                          'PRIMEIRO_CANAL_ACESSADO',\
                          'REGIAO_UF'\
                          'TIPO_CANAL',\
                          'ULTIMO_CANAL_ACESSADO',\
                          'ANALISE_CREDITO_APROVADO',\
                          'ULTIMO_LEAD_QTD_PARCELA',\
                          'ESTADO_CIVIL',\
                          'FEZ_ANALISE_CREDITO'
                        ]]
        X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=gen_params['seed']) 
        x_test, X_calib, y_test, y_calib = train_test_split(X_test, y_test, test_size=0.2, stratify=y_test, random_state=gen_params['seed'])
        return X_train, X_test, X_calib, y_train, y_test, y calib
    
    #DEFINICAO DA FUNCAO OBJETIVO: Funcao que vamos minimizar
    def obj_fun(self, params, model_dic, gen_params, X_train, y_train, scorer):
        scorer_name, scorer_metric = scorer 
        model = model_dic['estimator'].set_params(random_state = gen_params['seed'], **params)
        cv = StratifiedKFold(n_splits=gen_params['k_fold_split_hyper'], shuffle-True, 
                             random_state=gen_params['seed']) 
        scores = cross_validate(model, X_train, y_train, scoring=scorer_metric, cv=cv, 
                                return_train_score=True, n_jobs=-1) 
        best_score = np.mean(scores['test_score'])
        
        if scorer_name == 'log_loss':
            loss = -1*best_score 
            best_score= -1*best_score
        else: 
            loss = 1 - best_score
        
        if loss < self.best_loss: 
            self.best_loss = loss
            self.best_iter = self.iteracao + 1
        
        self.iteracao += 1
        print(' ---Modelo ' + model_dic['name'] + ' Iteracao: ' + str(self.iteracao) + '---')
        print('    Melhor score da iteracao: ' + scorer_name + ' = ' + str(best_score))
        print('    Best so far: iteracao ' + str(self.best_iter) + ' (' + str(self.best_loss) +')') 
        
        return {'loss': loss, 'status': STATUS_OK, 'model_name': model_dic['name'], 
                'params': params, 'scores': scores, 'metric': scorer_name, 'best_loss':self.best_loss}
    
    #OTIMIZACAO DA FUNCAO OBJETIVO UTILIZANDO HYPEROPT 
    def optimize(self, obj_fun_no_score, params, init_tpe, max_evals_hyper, rstate, scoring_dic, model_name): 
        best_dic = {}
        trials_dic = {}
        for scorer in scoring_dic.items():
            self.reinit_counters() 
            key, value = scorer
            print(' --- OTIMIZANDO A METRICA: '+ key + ' --------------------------------------------------') 
            algo = partial(tpe.suggest, n_startup_jobs=init_tpe)
            trials = Trials() 
            obj_fun = partial(obj_fun_no_score, scorer=scorer)
            best = fmin(obj_fun, params, algo=algo, max_evals=max_evals_hyper, trials=trials, rstate=rstate,
                        show_progressbar=False) 
            best_dic[model_name + '_' + key] = best
            trials_dic[model_name + '_' + trials.results[0]['metric']] = trials
            print()
            return best_dic, trials_dic
        
    #AVALIACAO DOS MODELOS TREINADOS
    def evaluate(self, gen_params, optmzd_model_data, model_dic, scoring_dic, X_train, y_train, x_test, y_test):
        model name, optimization_data_dic = optmzd_model_data
        print(' METRICAS DE TREINAMENTO: modelo ' + model_name)
        results_dic = self.calc_training_results(gen_params, model_name, optimization_data_dic, scoring_dic)
        best fitted_model= self.fit_best_model_to_tr_data(gen_params, model_dic[model_name], model_name, results_dic, X_train, y_train)
        self.calc_tr_curves (gen_params, results_dic, model_name, best_fitted_model, X_test, y_test)
        print(' AVALIAÇÃO DO MODELO COM HIPERPARAMETROS OTIMIZADOS: modelo ' + model_name) 
        self.calc_classif_metrics(gen_params, results_dic, model_name, best_fitted_model, x_test, y_test) 
        pred results, evaluation_df = self.calc_pred_results(gen_params, results_dic, model_name, 
                                                             best_fitted_model, x_test, y_test)
        self.evaluate_probb_calib (gen_params, model_name, pred_results, evaluation_df)
        self.calc_histogram (gen_params, model_name, pred_results)
        self.calc kde_intersection(gen_params, model_name, pred_results)         
        print(" TABELA DE RESUMO: modelo + model_name) 
        self.summary_table()
              
    def calc_training results(self, gen_params, model_name, optimization data, scoring_dic):
        results_df_dic = {}
        best_loss = 1
        best_trial = Trial()
        best_metric = ''  
        best_params = {}              
        for metric, scorer in scoring_dic.items():
            results = optimization_data['trials'][model_name+'_'+metric].results            
            if results['best_loss'] < best_loss:
                best_loss =  results['best_loss'] 
                best_trial = optimization_data['trials'][model_name+'_' +metric]
                best_metric = metric
                best_params = optimization_data['best_hps'][model_name+'_'+metric]
              
        results = best_trial.results
        results_df pd.DataFrame({ 'loss': [x['loss'] for x in results], 
                                 'params': [x['params'] for x in results],
                                 'scores': [x['scores'] for x in results], 
                                 'best score': [np.mean(x['scores']['test_score']) for x in results]
                                })
              
        results_df = results_df.sort_values('loss', ascending = True).reset_index() 
        results_df_dic = {'model': results[0]['model_name'], 'results' :results_df, 'metric': best_metric, 
                          'best_params': best_params] 
        
        joblib.dump(results_df_dic, gen_params['saving_path'] + results[0]['model_name'] + 'dic_tr_res_' + str(gen_params['max_evals_hyper'])+ 
                    gen_params['mdl_version'] + gen_params['lead_domain'] +'.jblib') 
        print(" Calculados os resultados de treinamento...') 
        return results_df_dic
              
    def fit_best_model_to_tr_data(self, gen_params, model_dic, model_name, results_dic, X_train, y_train):
        best_params = results_dic[model_name]['best_params'] 
        best fitted_model= model_dic['estimator'].set_params(random_state = gen_params['seed'], **best_params) 
        best fitted_model.fit(np.array (X_train), np.array(y_train))
        joblib.dump(best_fitted_model, gen_params['saving_path'] + model_name + estim_best_fitted + 
                    str(gen_params['max_evals_hyper']) + gen_params['mdl_version'] + 
                    gen_params['lead_domain'] + '.jblib')
        print(' Melhor conjunto de hiperparametros utilizado para fitar o modelo ao conjunto de dados de treinamento...') 
        return best_fitted_model

    def calc_tr_curves(self, gen_params, results, model name, best_fitted_model, X_test, y_test):
        results_scores = results['results']['scores'] 
        epochs = gen_params['max_evals_hyper']
        x_axis = list(range(0, epochs))

        #plot Log Loss
        fig1, ax1 = plt.subplots(figsize=(12,12))
        ax1.plot(x_axis, [-np.mean(scores['train_neg_log_loss']) for scores in results_scores], 
                 label='Train')
        ax1.plot(x_axis, [-np.mean(scores['test neg_log_loss']) for scores in results_scores],
                 label='Test')
        ax1.legend() 
        plt.ylabel('Log Loss') 
        plt.title(results['model'] + ' Log Loss') 
        filename = gen_params['saving_path'] + model_name + '_img_logloss_tr_curve_' + 
              str(gen_params['max_evals_hyper'])+ gen_params['mdl_version'] + gen_params['lead_domain'] + 
              '.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0)
              
        # plot ROC AUC
        fig2, ax2 = plt.subplots(figsize=(12,12))
        ax2.plot(x_axis, [np.mean(scores['train_roc_auc']) for scores in results_scores], label='Train')
        ax2.plot(x_axis, [np.mean(scores['test_roc_auc']) for scores in results_scores], label='Test')
        ax2.legend()
        plt.ylabel('ROC AUC');
        plt.title(results['model'] + 'Area Under ROC Curve')
        filename = gen_params['saving_path'] + model_name + '_img_auc_tr_curve_' + 
              str(gen_params['max_evals_hyper']) + gen_params['mdl_version'] + gen_params['lead_domain']
              + '.png'
        plt.savefig(filename, dpi=300, bbox_inches= 'tight', pad_inches=0)
              
        # plot matthews
        fig3, ax3 = plt.subplots(figsize=(12,12))
        ax3.plot(x_axis, [np.mean(scores['train_matthews']) for scores in results_scores], label='Train')
        ax3.plot(x_axis, [np.mean(scores['test_matthews']) for scores in results_scores], label='Test')
        ax3.legend() 
        plt.ylabel('Matthews Correlation Coefficient')
        plt.title(results['model'] + ' Matthews Correlation Coefficient')
        filename = gen_params['saving_path'] + model_name + '_img_logloss_tr_curve_' + 
              str(gen_params['max_evals_hyper']) + gen_params['md1_version'] + gen_params['lead_domain'] +
              '.png' 
        plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0)
                      
        plot_roc_curve(best_fitted_model, np.array(x_test), np.array(y_test)) 
        filename = gen_params['saving_path'] + model_name + '_img_ roc_curve_' + 
              str(gen_params['max_evals_hyper'])+ gen_params['mdl_version'] + gen_params['lead_domain'] + 
              '.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0)
        print(' Plotadas as curvas de treinamento...')
              
    def calc_classif_metrics(self, gen_params, results, model_name, best_model, X_test, y_test):
        cm = confusion_matrix(y_test, best_model.predict(np.array(x_test)), labels=[0, 1]) 
        ax=plt.subplot() 
        sns.heatmap(cm, annot=True, ax=ax, fmt='g' , cmap="Blues")
        ax.set_xlabel('Preditos') 
        ax.set_ylabel('Reais')
        x.set_title('Matriz de Confusão') 
        ax.xaxis.set_ticklabels(['Não Pago', 'Pago'])
        filename = gen_params['saving_path'] + model_name + '_img_conf_mat_'+ str(gen_params['max_evals_hyper']) + gen_params['mdl_version'] + gen_params['lead_domain'] + '.png' 
        plt.savefig(filename, dpi=300, bbox_inches='tight', pad_inches=0)
        
        class_rep_str = classification_report(y_test, best_model.predict(np.array(X_test)), 
                                              target_names=['Não Pago', 'Pago']) 
        class_rep_dict = classification_report(y_test, best_model.predict(np.array(x_test)), 
                                               target_names=['Não Pago', 'Pago'], output_dict=True)
        joblib.dump(class_rep_dict, gen_params['saving_path'] + model_name + '_dic_classif_report_' + 
                    str(gen_params['max_evals_hyper']) + gen_params['md1_version'] + gen_params['lead_domain']
                    + '.jblib') 
        print('- Calculadas as métricas de classificação no conjunto de dados de teste...')
        return class_rep_str
              
    def calc_pred_results(self, gen_params, results, model_name, best_model, X_test, y_test): 
        dic = {'y_test': y_test, 'y_pred': best_model.predict (np.array(x_test)), 'y_pred_prob': best_model.predict_proba(np.array(x_test))[:, 1]}
        df_model_results = pd.DataFrame(data=dic) 
        bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
        labels = ['0 a 10', '10 a 20', '20 a 30', '30 a 40', '40 a 50', '50 a 60', '60 a 70', '70 a 80', '80 a 90', '90 a 100'] 
        df_model_results['faixa_score'] = pd.cut(df_model_results['y_pred_prob'], bins=bins, labels=labels) 
        df_model_results['score'] = df_model_results.y_pred_prob.apply(lambda x: round (x*100)) 
        joblib.dump(df_model_results, gen_params['saving_path'] + model_name + '_df_pred_results_' + str(gen_params['max_evals_hyper']) + gen_params['mdl_version'] + gen_params['lead_domain'] + '.jblib')
        
        df_evaluation = pd.concat([df_model_results.groupby(['faixa_score']).count()[['y_test']],
                                   df_model_results.groupby(['faixa_score']).count()[['y_test' ]]/df_model_results['y_test'].count(),
                                   df_model_results.groupby(['faixa_score']).sum()[['y_test']],
                                   df_model_results.groupby(['faixa_score']).sum()[['y_test']]/df_model_results.groupby(['faixa_score']).count()[['y_test']], 
                                   df_model_results.groupby(['faixa_score']).count()[['y_test']] - df_model_results.groupby(['faixa_score']).sum()[['y_test']],
                                   df_model_results.groupby(['faixa_score']). sum() [['y_test']]/df_model_results['y_test'].sum()], 
                                  axis=1).reset_index()
        df_evaluation.columns = ['faixa score', 'leads', '%(total)', 'pagos', 'conversão faixa', 'não pagos', 'conversão geral'] 
        df_evaluation['conversão faixa'].fillna(0, inplace=True)
        df_evaluation['conversão geral'].fillna(0, inplace=True) 
        total = dict('faixa score': 'Totais, 
                     'leads': df_evaluation['leads'].sum(),
                     '%(total)': 1,
                     'pagos': df_evaluation['pagos'].sum(),
                     'conversão Faixa': (df_evaluation['pagos'].sum()/df_evaluation['leads'].sum()),
                     'não pagos': df_evaluation['não pagos'].sum(), 
                     'conversão geral': df_evaluation['conversão geral'].sum())
        totais = pd.Series(data=total_dict, name=11) 
        df_evaluation = df_evaluation.append(totais) 
        df_evaluation['%(total)'] = pd.Series(['[0:.2f}%'.format(val*100) for val in df_evaluation['%(total)']], 
                                              index =df_evaluation.index)
        df_evaluation['pagos'] = pd.Series(['(0:.0f)'.format(val) for val in df_evaluation['pagos']], 
                                           index = df_evaluation.index)
        df_evaluation['conversão faixa'] = pd.Series(['(0:.2f}%'.format(val * 100) for val in df_evaluation['conversão faixa']], 
                                                     index=df_evaluation.index) 
        df_evaluation['não pagos'] = pd.Series(["(0:.0f)".format(val) for val in df_evaluation['não pagos']], index = df_evaluation.index) 
        df evaluation ['conversão geral'] = pd.Series(['0:.2f%'.format(val * 100) for val in df_evaluation['conversão geral']], 
                                                      index = df_evaluation.index)
        joblib.dump(df_evaluation, gen_params['saving path'] + model_name + '_df_evaluation_' + str(gen_params['max_evals_hyper']) + gen_params['mdl_version'] + gen_params['lead_domain'] + '.jblib')
        print(" Calculados os resultados de conversão para avaliação do modelo...")
        return df_model_results, df_evaluation

        def calc_histogram(self, gen_params, model_name, pred_results):
            sns.set_style('whitegrid')
            fig, ax. plt.subplots()
               